# 05 · Mini Agent Loop

So far we handled **one round** of tools by hand. But what if the model needs a tool, reads the result, and *then* needs another tool?

An agent is just a **loop**:

```
        ┌─────────────────────────────────────────┐
        ▼                                         │
   Ask the model ──> Did it ask for tools? ──yes──> Run them, add results
                              │
                              no
                              ▼
                        Final answer
```

This is the Think, Act, Observe loop from the concepts page, in about 20 lines of Python.

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()   # read your keys from the .env file
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to your .env file (see .env.example)"

client = OpenAI()        # finds OPENAI_API_KEY on its own
MODEL = "gpt-4o-mini"    # a small, cheap model is plenty for this module
print("✅ Ready")

✅ Ready


In [2]:
import requests

def get_weather(city):
    """Get the current weather for a city using the free Open-Meteo API (no API key needed)."""

    # Step 1: turn the city name into latitude and longitude
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
        timeout=10,
    ).json()
    if "results" not in geo:
        return f"Sorry, I could not find a city called {city}."
    place = geo["results"][0]

    # Step 2: ask for the weather at that location
    data = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "current": "temperature_2m,precipitation",
        },
        timeout=10,
    ).json()
    now = data["current"]

    return f"{place['name']}: {now['temperature_2m']}°C, rain right now: {now['precipitation']} mm"

In [3]:
from datetime import date

def get_today():
    """Return today's date. A model does not know what day it is, so this is a handy tool."""
    return date.today().isoformat()

def percentage_of(percent, amount):
    """Work out a percentage, for example 18% of 2400."""
    return percent / 100 * amount

# The descriptions below are what the model reads to decide WHICH tool to use,
# so write them clearly.
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather (temperature and rain) for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name, for example Pune"},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_today",
            "description": "Get today's date.",
            "parameters": {"type": "object", "properties": {}},   # this tool needs no input
        },
    },
    {
        "type": "function",
        "function": {
            "name": "percentage_of",
            "description": "Calculate a percentage of an amount, for example 18 percent of 2400.",
            "parameters": {
                "type": "object",
                "properties": {
                    "percent": {"type": "number", "description": "The percentage, for example 18"},
                    "amount": {"type": "number", "description": "The amount to take the percentage of"},
                },
                "required": ["percent", "amount"],
            },
        },
    },
]

# A lookup table: tool name -> the real Python function
available_tools = {
    "get_weather": get_weather,
    "get_today": get_today,
    "percentage_of": percentage_of,
}

def run_tool(name, args):
    """Run the tool the model asked for and return the result as text."""
    if name not in available_tools:
        return f"Unknown tool: {name}"
    try:
        return str(available_tools[name](**args))
    except Exception as error:      # if a tool breaks, tell the model instead of crashing
        return f"The tool failed: {error}"

## The agent
Read it from top to bottom. The `for` loop is the agent. Each lap is one call to the model.

`max_steps` is a safety limit. Without it, a confused model could loop forever and every lap costs money.

In [4]:
def run_agent(question, max_steps=5):
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use the tools when you need them."},
        {"role": "user", "content": question},
    ]

    for step in range(1, max_steps + 1):
        # THINK: ask the model what to do next
        response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        message = response.choices[0].message
        messages.append(message)

        # No tool requested? The model has its answer, so we are done.
        if not message.tool_calls:
            return message.content

        # ACT + OBSERVE: run each requested tool and add the result to the conversation
        for tool_call in message.tool_calls:
            args = json.loads(tool_call.function.arguments)
            result = run_tool(tool_call.function.name, args)
            print(f"Step {step}: {tool_call.function.name}({args}) -> {result}")

            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

    return "Stopped: too many steps."

## Try it
A question that needs the weather **and** a calculation:

In [5]:
answer = run_agent("Should I carry an umbrella in Pune today? Also, what is 18% of 2400?")
print("\nAnswer:", answer)

Step 1: get_weather({'city': 'Pune'}) -> Pune: 18.6°C, rain right now: 0.0 mm
Step 1: percentage_of({'percent': 18, 'amount': 2400}) -> 432.0



Answer: The current weather in Pune is 18.6°C with no rain right now, so you likely do not need to carry an umbrella today. 

Additionally, 18% of 2400 is 432.


Now a question where the second tool needs **the result of the first**. The model cannot calculate 10% of the temperature until it knows the temperature, so it needs **two laps** of the loop. Watch for `Step 1` and `Step 2`.

In [6]:
answer = run_agent("What is 10% of the current temperature in Pune?")
print("\nAnswer:", answer)

Step 1: get_weather({'city': 'Pune'}) -> Pune: 18.6°C, rain right now: 0.0 mm


Step 2: percentage_of({'percent': 10, 'amount': 18.6}) -> 1.8600000000000003



Answer: 10% of the current temperature in Pune, which is 18.6°C, is approximately 1.86°C.


## Your turn
Ask your own question, or add a fourth tool. Copy the pattern: a Python function, a description in `tools`, and an entry in `available_tools`.

In [7]:
# Change this question to anything you like
answer = run_agent("Is it warmer in Pune or in Mumbai right now?")
print("\nAnswer:", answer)

Step 1: get_weather({'city': 'Pune'}) -> Pune: 18.6°C, rain right now: 0.0 mm


Step 1: get_weather({'city': 'Mumbai'}) -> Mumbai: 23.1°C, rain right now: 0.0 mm



Answer: Currently, Mumbai is warmer at 23.1°C, while Pune is at 18.6°C.


## Recap and what's next
* An agent is a loop: **ask the model, run the tools it requests, repeat until it answers.**
* `max_steps` keeps the loop (and your bill) under control.
* Everything here uses plain Python and one OpenAI call. Frameworks like LangChain or LangGraph wrap this same loop with extra features.

**Next: Module 03, Agent Memory.** Right now every `run_agent` call starts from scratch. Next the agent will remember earlier conversations and look things up in your documents.